## 4.4 RNN网络结构 - 完整网络结构与折叠图、展开图

#### 1、这一节我们要解决什么问题 🎯

在上一小节中，我们已经知道了 RNN 在每一个时间步都会做这些事：

- 接收当前输入 $x_t$
- 接收上一时刻的隐藏状态 $h_{t-1}$
- 计算当前隐藏状态 $h_t$
- 根据 $h_t$ 得到当前输出 $y_t$

##### 1.1 本节核心问题

1. RNN 的“模型网络结构”到底长什么样？
2. 为什么有时候图里画成一个“自己连自己”的结构？
3. 为什么有时候又画成很多个一模一样的小块排成一行？
4. 折叠图和展开图，到底是在画“不同模型”，还是在画“同一个模型的不同表示方式”？
5. 这些图，和我们上一节学到的公式到底怎么对应？

这一节最重要的核心认识是：

RNN 层只是特征提取器，它的隐藏状态不是最终输出；最终输出需要再接一个全连接层（FC层）来完成。

#### 2、折叠图和展开图 ✅

##### 2.1 最核心的结论

折叠图和展开图，不是两个不同的模型。它们表示的都是同一个 RNN 模型，区别只在于：

- 折叠图：把时间过程压缩起来画，强调“有循环”，是结构摘要图
- 展开图：把时间过程摊开来画，强调“怎么一步一步运行”，是运行流程图

它们只是同一个 RNN 结构的两种表示方式。

它们都对应同一组公式：

$h_t = f(W_x \cdot x_t + W_h \cdot h_{t-1} + b)$

$y_t = g(W_y \cdot h_t + b_y)$

##### 2.2 RNN 的真正结构，不在“图长什么样”，而在“计算关系是什么”

RNN 真正的结构核心不是画图本身，而是下面这个关系：

$h_t = f(W_x x_t + W_h h_{t-1} + b)$

$y_t = g(W_y h_t + b_y)$

这两个公式说明了：

- 当前时刻要接收当前输入 $x_t$
- 当前时刻要接收上一时刻状态 $h_{t-1}$
- 它们共同决定当前状态 $h_t$
- 当前状态再决定输出 $y_t$

这才是 RNN 的结构本体。

图只是把这个结构画出来而已。

##### 2.3 什么是折叠图（Folded Graph）

折叠图通常会把 RNN 计算单元画成一个小模块，然后有一条箭头从模块的隐藏状态输出，再绕回模块的输入附近。

折叠图想表达的核心：这个单元不是只计算一次，而是会反复使用；并且前一次产生的状态，会影响下一次计算。

特别注意：折叠图里的“环”代表的是跨时间的状态传递，而不是同一时刻内部反复循环。

所以你可以把折叠图理解成：

RNN 结构的“压缩版示意图”

折叠图回答的问题是：

“这个网络为什么叫循环神经网络？”

##### 2.4 什么是展开图（Unrolled Graph）

展开图就是把折叠图中的“循环过程”沿着时间轴拆开。比如一个序列有 3 个时间步 $x_1, x_2, x_3$，展开后就变成 3 个排列的计算单元，每个时间步做同样的事：

第一步：$x_1 + h_0 \rightarrow h_1 \rightarrow y_1$

第二步：$x_2 + h_1 \rightarrow h_2 \rightarrow y_2$

第三步：$x_3 + h_2 \rightarrow h_3 \rightarrow y_3$

展开图不是新模型。这 3 个模块用的是同一个 RNN，同一组 $W_x$、$W_h$、$W_y$，不同的只是当前输入和当前状态。

展开图回答的问题是：

“这个网络处理序列时，到底是怎么一步一步跑的？”

##### 2.5 理解 RNN 的两个维度

理解 RNN 结构时，需要同时从两个维度来看：

**（1）时间维度（展开图体现）**

$h_0 \rightarrow h_1 \rightarrow h_2 \rightarrow h_3 \dots$

前一时刻状态传给后一时刻，形成递推链条。

**（2）单步内部的网络维度（每个小方块内部）**

输入 $x_t + h_{t-1} \rightarrow$ 加权求和与激活 $\rightarrow$ 新状态 $h_t \rightarrow$ 输出 $y_t$

所以：

RNN = 神经网络模块 + 时间上的循环递推结构


#### 3、关键认识：RNN 层和完整模型的区别 ⭐

这是这一节最容易忽视、也最容易混淆的地方。

##### 3.1 RNN 层只是“特征提取器”

在展开图里，每个时间步都会输出一个 $y_t$，或者产生一个隐藏状态 $h_t$。

但这里要特别注意：

RNN 层（`nn.RNN`）本身输出的是隐藏状态，而不是最终的分类结果或预测结果。

隐藏状态是一个特征向量，它是对序列信息的“压缩表达”，还不是我们最终想要的答案。

这一点和 CNN 完全类比：

- CNN 中：Conv 层提取空间特征 $\rightarrow$ 最后接 FC 层 $\rightarrow$ 输出分类结果
- RNN 中：RNN 层提取时序特征 $\rightarrow$ 最后接 FC 层 $\rightarrow$ 输出分类/预测结果

RNN 层的作用 = CNN 中卷积层的作用：提取特征，而不是直接输出答案。

##### 3.2 为什么还需要全连接层

RNN 层输出的隐藏状态 $h_t$，维度是 `hidden_size`，例如 256 维。

但我们最终想要的输出，往往是：

- 分类任务：输出 `num_classes` 个类别的概率（如 10 类）
- 回归任务：输出一个或多个数值

所以需要一个全连接层（Linear 层）来做这个映射：

$h_{last} \rightarrow Linear(hidden\_size, num\_classes) \rightarrow 输出\ logits$

全连接层负责把 RNN 层提取到的时序特征，转化为最终的任务输出。

##### 3.3 完整 RNN 模型的结构

一个完整的 RNN 模型通常长这样：

输入序列 $\rightarrow$ RNN层（提取时序特征）$\rightarrow$ 取隐藏状态 $\rightarrow$ 全连接层（FC）$\rightarrow$ 输出结果

用 PyTorch 描述就是：

$x \rightarrow nn.RNN \rightarrow output / hn \rightarrow nn.Linear \rightarrow \hat{y}$

其中：

- `nn.RNN` 负责处理整个序列，输出每个时间步的隐藏状态
- `nn.Linear` 负责把最后的隐藏状态映射到目标维度

##### 3.4 取哪个时间步的隐藏状态接 FC 层？

这取决于任务类型（详细内容见 1.4 节），简单来说：

- $N:1$ 任务（如文本分类）：取最后一步 $h_T$，接 FC 层输出一个结果
- $N:N$ 任务（如序列标注）：每一步的 $h_t$ 都接 FC 层，输出每步的结果

最常见的情况（分类任务）是：

```python
output, hn = rnn(x)      # RNN层输出
h_last = hn[-1]          # 取最后一层的最终隐藏状态
y = fc(h_last)           # 接全连接层得到输出
```

#### 4. 展开图 + 完整模型结构的统一理解 🌰
以文本分类任务为例，序列 `x_1, x_2, x_3`：

##### 4.1 第一阶段：RNN 层（时序特征提取）
* 第一步：x_1 + h_0 → h_1（还在提取特征，不输出结果）
* 第二步：x_2 + h_1 → h_2（继续更新记忆）
* 第三步：x_3 + h_2 → h_3（读完整个序列，h_3 融合了全序列信息）

##### 4.2 第二阶段：全连接层（输出映射）
* h_3 → Linear → ŷ（分类结果）
这里的关键：h_3 只是一个特征向量（例如 256 维），还不是答案；经过 FC 层之后，才变成我们想要的结果（例如 10 个类别的概率）。

##### 4.3 参数共享的本质
虽然展开后像有 3 个模块，但它们都是同一个 RNN 单元，用同一组参数：
* 同一个 W_x、W_h（RNN 层参数）
* 同一个 W_fc（全连接层参数）
展开图只是把同一个 RNN 沿时间多次使用的过程画出来而已。


#### 5. 本节总结 🧾
* 折叠图和展开图描述的是同一个 RNN 模型，只是表达角度不同
* 折叠图 = 结构摘要图（强调有循环）；展开图 = 运行流程图（强调时间递推）
* 折叠图里的"环"代表跨时间的状态传递，不是原地打转
* RNN 层只是特征提取器，输出的是隐藏状态，不是最终答案
* 完整 RNN 模型 = RNN层（时序特征提取）+ 全连接层（输出映射）
* 这一点和 CNN 完全类比：Conv层 → FC层 ≈ RNN层 → FC层
* 具体取哪个时间步的隐藏状态接 FC 层，由任务类型（N:1 or N:N）决定，详见 2.3 节